# A1 — Kako preverimo model: Telekom

Nadaljevanje `A1_Priprava_Telekom.ipynb`. Pripravljeno tabelo razdelimo v razmerju
**60 % / 20 % / 20 %** in nato dopolnimo manjkajoče vrednosti in besedilne stolpce
spremenimo v stolpce 0/1. Delitev in pravila
shranimo za vse naslednje modele. Napovednega modela še ne učimo.

Ena vrstica je naročnik. Za ta učni primer uporabimo stratificirano naključno
delitev; izvoz nima datumov, s katerimi bi lahko preverili časovno posploševanje.

Podatke po potrebi pripravi `poskrbi_za_modeliranje`; `preberi_tabelo` jih bere s tipi iz sheme.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from pathlib import Path
import json
import hashlib
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

IME = "telekom"
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
VHOD = poskrbi_za_modeliranje(IME, delitve=False)
pot_csv = VHOD / f"{IME}_modeliranje.csv"
pot_sheme = VHOD / f"{IME}_shema.json"
shema = preberi_shemo(pot_sheme)
ZNACILKE = shema["znacilke"]
NUM = shema["numericne_znacilke"]
KAT = shema["kategoricne_znacilke"]
META, CILJ, KLJUC = shema["metapodatki"], shema["cilj"], shema["kljuc"]
DATUMI = shema["datumski_stolpci"]
podatki = preberi_tabelo(pot_csv, shema)
assert podatki[KLJUC].notna().all() and podatki[KLJUC].is_unique
assert podatki[CILJ].notna().all(), "Cilja ne dopolnjujemo."
assert not set(META + [CILJ]) & set(ZNACILKE)
assert set(NUM).isdisjoint(KAT) and set(NUM + KAT) == set(ZNACILKE)
assert not np.isinf(podatki[NUM].to_numpy(dtype=float)).any()
IZHOD = VHOD / "delitve" / IME
IZHOD.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_rows", 12)
pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (9, 3.5), "font.size": 11})
print(f"Vhod: {len(podatki):,} vrstic, {len(ZNACILKE)} značilk.")

## 1. Najprej delitev, nato računanje median
Najprej odložimo 40 % vrstic, nato ta del razdelimo na polovico. `stratify`
približno ohrani delež odhodov; `random_state=42` omogoči ponovitev iste delitve.
Validacijo uporabljamo za izbor, test pa odložimo za končno oceno.

In [ ]:
from sklearn.model_selection import train_test_split
SEME = 42
DELEZ_TRAIN, DELEZ_VAL, DELEZ_TEST = 0.60, 0.20, 0.20
assert np.isclose(DELEZ_TRAIN + DELEZ_VAL + DELEZ_TEST, 1)
assert min(DELEZ_TRAIN, DELEZ_VAL, DELEZ_TEST) > 0
# Stabilen vrstni red pomeni, da vhodno preurejanje CSV-ja ne spremeni delitve.
podatki = podatki.sort_values(KLJUC).reset_index(drop=True)
train_i, ostalo_i = train_test_split(podatki.index,
    test_size=DELEZ_VAL + DELEZ_TEST, stratify=podatki[CILJ], random_state=SEME)
val_i, test_i = train_test_split(ostalo_i,
    test_size=DELEZ_TEST / (DELEZ_VAL + DELEZ_TEST),
    stratify=podatki.loc[ostalo_i, CILJ], random_state=SEME)
vloge = pd.Series("", index=podatki.index, dtype="string")
for ime, indeksi in [("train", train_i), ("val", val_i), ("test", test_i)]:
    vloge.loc[indeksi] = ime
razlogi = pd.Series("vkljuceno", index=podatki.index, dtype="string")
PARAMETRI_DELITVE = {"metoda": "stratificirana_nakljucna", "seme": SEME,
    "delezi": {"train": DELEZ_TRAIN, "val": DELEZ_VAL, "test": DELEZ_TEST}, "stratifikacija": CILJ}
assert vloge.ne("").all()

## 2. Preverimo ločenost in obseg zbirk
Vsak naročnik je v natanko eni zbirki. Preverimo velikosti; delež odhodov
pokažemo samo za učni in validacijski del. Testnih rezultatov ne analiziramo.

In [ ]:
# Pripadnost določa vsak naslednji poskus, ne izbrani model.
zbirke_pred = {ime: podatki.loc[vloge.eq(ime)].copy() for ime in ["train", "val", "test"]}
assert all(len(df) > 0 for df in zbirke_pred.values())
kljuci = {ime: set(df[KLJUC]) for ime, df in zbirke_pred.items()}
assert kljuci["train"].isdisjoint(kljuci["val"])
assert kljuci["train"].isdisjoint(kljuci["test"])
assert kljuci["val"].isdisjoint(kljuci["test"])
assert len(vloge) == len(podatki) and vloge.notna().all()
assert set(vloge.unique()) <= {"train", "val", "test", "izloceno"}
assert sum(len(df) for df in zbirke_pred.values()) + vloge.eq("izloceno").sum() == len(podatki)

evidenca = podatki[META].copy()
evidenca["zbirka"] = vloge
evidenca["razlog"] = razlogi
pregled = pd.DataFrame([
    {"zbirka": ime, "vrstic": len(df), "delež vseh (%)": 100 * len(df) / len(podatki)}
    for ime, df in zbirke_pred.items()
])
display(pregled.round(2))

In [ ]:
display(pd.DataFrame([
    {"zbirka": ime, "delež odhodov (%)": 100 * zbirke_pred[ime][CILJ].mean()}
    for ime in ["train", "val"]
]).round(2))
ax = pregled.plot.bar(x="zbirka", y="vrstic", legend=False, rot=0,
    color=["#205781", "#2993AD", "#94AD35"])
ax.set(title="Telekom: ista delitev za vse modele", xlabel="", ylabel="Število naročnikov")
plt.tight_layout(); plt.show()

## Manjkajoče vrednosti: naučimo se samo iz učne množice
Za numerične značilke izračunamo **učne mediane**. Kategorije dopolnimo s stalno
oznako `NEZNANO`; že znanih kategorij ne spreminjamo. Kazalnike manjkanja iz
prvega zvezka ohranimo. Ista pravila uporabimo na vseh treh zbirkah.

Če bi bil številčni stolpec v učni množici v celoti prazen, mediana ne bi obstajala:
izpišemo ga in uporabimo vnaprej dogovorjeno nadomestilo 0. To je tehnična
nadomestna vrednost, ne trditev o resnični meritvi. Stolpcev ne odstranimo.

In [ ]:
def nauci_dopolnjevanje(X_train, numericni, kategoricni):
    """Vrednosti za dopolnjevanje določimo samo iz podanega učnega X."""
    mediane = X_train[numericni].astype(float).median()
    prazni = mediane.index[mediane.isna()].tolist()
    return {
        "numericni": list(numericni), "kategoricni": list(kategoricni),
        "mediane": mediane.fillna(0.0).to_dict(),
        "povsem_prazni_ucni_stolpci": prazni,
        "nadomestilo_za_prazen_numericni_stolpec": 0.0,
        "neznana_kategorija": "NEZNANO",
    }

def dopolni(X, pravila):
    """Uporaba že določenih pravil; nobene nove mediane ali drugega učenja."""
    rezultat = X.copy()
    for c in pravila["numericni"]:
        rezultat[c] = rezultat[c].astype(float).fillna(pravila["mediane"][c])
    for c in pravila["kategoricni"]:
        rezultat[c] = rezultat[c].astype("string").fillna(pravila["neznana_kategorija"])
    return rezultat

pravila = nauci_dopolnjevanje(zbirke_pred["train"][ZNACILKE], NUM, KAT)
print("V celoti prazni učni numerični stolpci:", pravila["povsem_prazni_ucni_stolpci"])
ucni_pregled = pd.DataFrame({
    "manjka v učni": zbirke_pred["train"][NUM].isna().sum(),
    "učna mediana / nadomestilo": pd.Series(pravila["mediane"]),
})
display(ucni_pregled.loc[ucni_pregled["manjka v učni"].gt(0)].sort_values("manjka v učni", ascending=False))

In [ ]:
pravila_pred = copy.deepcopy(pravila)
zbirke = {}
kontrola = []
for ime, df in zbirke_pred.items():
    X_pred = df[ZNACILKE]
    X_po = dopolni(X_pred, pravila)
    assert X_po.index.equals(X_pred.index) and X_po.columns.tolist() == ZNACILKE
    assert not X_po.isna().any().any()
    assert np.isfinite(X_po[NUM].to_numpy(dtype=float)).all()
    # Dopolnili smo samo manjkajoče vrednosti, opazovanih meritev nismo spreminjali.
    for c in NUM:
        znano = X_pred[c].notna()
        np.testing.assert_allclose(X_po.loc[znano, c], X_pred.loc[znano, c], rtol=0, atol=0)
    for c in KAT:
        znano = X_pred[c].notna()
        assert X_po.loc[znano, c].astype("string").equals(X_pred.loc[znano, c].astype("string"))
    zbirke[ime] = pd.concat([df[META + [CILJ]], X_po], axis=1)
    pd.testing.assert_frame_equal(zbirke[ime][META + [CILJ]], df[META + [CILJ]])
    kontrola.append({"zbirka": ime, "manjkajočih celic pred": int(X_pred.isna().sum().sum()),
                     "manjkajočih celic po": int(X_po.isna().sum().sum())})
assert pravila == pravila_pred, "Uporaba pravil ne sme spreminjati učnih vrednosti."
display(pd.DataFrame(kontrola))

## Besedilni stolpci: kategorije spremenimo v stolpce 0/1
Modeli v scikit-learn računajo s številkami. Vsaka kategorija zato dobi svoj stolpec:
1 = vrstica spada v to kategorijo, 0 = ne spada (*one-hot* kodiranje).

Seznam kategorij določimo **samo iz učne množice**. Kategorija z manj kot 100 učnimi
vrsticami ima premalo primerov za zanesljivo pravilo, zato jo štejemo v skupni stolpec
`DRUGO`. Tja gre tudi kategorija, ki je v učni množici ni.

Besedilne stolpce ohranimo za branje, modelom pa podamo stolpce 0/1. Številskih značilk
tukaj ne skaliramo: drevesa skaliranja ne potrebujejo, pravila pa ostanejo v pravih enotah.
Model, ki skaliranje potrebuje, ga naredi v svojem zvezku, prav tako samo iz učne množice.

In [ ]:
import re

def nauci_kodiranje(X_train, kategoricni, najmanj=100):
    """Seznam kategorij določimo samo iz podanega učnega X."""
    pravila = {"najmanj_ucnih_vrstic": najmanj, "skupna_kategorija": "DRUGO", "stolpci": {}}
    for c in kategoricni:
        stevilo = X_train[c].astype("string").value_counts()
        pravila["stolpci"][c] = {
            "kategorije": sorted(stevilo.index[stevilo >= najmanj]),
            "v_skupni_kategoriji": sorted(stevilo.index[stevilo < najmanj]),
        }
    return pravila

def ime_kodiranega(stolpec, kategorija):
    # V imenu stolpca pustimo samo črke, številke in podčrtaje.
    return stolpec + "_" + re.sub(r"[^0-9A-Za-z]+", "_", kategorija).strip("_")

def kodiraj(X, pravila):
    """Uporaba že določenih kategorij; novih kategorij se ne učimo."""
    novi = {}
    for c, p in pravila["stolpci"].items():
        vrednosti = X[c].astype("string")
        for k in p["kategorije"]:
            novi[ime_kodiranega(c, k)] = vrednosti.eq(k).astype("int8")
        if p["v_skupni_kategoriji"]:
            novi[ime_kodiranega(c, pravila["skupna_kategorija"])] = (~vrednosti.isin(p["kategorije"])).astype("int8")
    return pd.DataFrame(novi, index=X.index)

kodiranje = nauci_kodiranje(zbirke["train"][ZNACILKE], KAT)
display(pd.DataFrame([
    {"besedilni stolpec": c, "kategorij v učni": len(p["kategorije"]) + len(p["v_skupni_kategoriji"]),
     "svoj stolpec 0/1": len(p["kategorije"]), "v skupnem stolpcu DRUGO": len(p["v_skupni_kategoriji"])}
    for c, p in kodiranje["stolpci"].items()
]))

In [ ]:
kodiranje_pred = copy.deepcopy(kodiranje)
kodirane = {ime: kodiraj(df[ZNACILKE], kodiranje) for ime, df in zbirke.items()}
KODIRANE = {c: [ime_kodiranega(c, k) for k in p["kategorije"] + [kodiranje["skupna_kategorija"]] * bool(p["v_skupni_kategoriji"])]
            for c, p in kodiranje["stolpci"].items()}
KODIRANE_VSE = [s for stolpci in KODIRANE.values() for s in stolpci]
assert KODIRANE_VSE == kodirane["train"].columns.tolist() and len(KODIRANE_VSE) == len(set(KODIRANE_VSE))
assert not set(KODIRANE_VSE) & set(META + [CILJ] + ZNACILKE), "Ime novega stolpca že obstaja."
kontrola = []
for ime, K in kodirane.items():
    assert K.columns.tolist() == KODIRANE_VSE and K.index.equals(zbirke[ime].index)
    assert K.isin([0, 1]).all().all()
    for c, p in kodiranje["stolpci"].items():
        vsota = K[KODIRANE[c]].sum(axis=1)
        # Vsaka vrstica ima največ eno enico na besedilni stolpec; s stolpcem DRUGO natanko eno.
        assert vsota.le(1).all() and (vsota.eq(1).all() or not p["v_skupni_kategoriji"])
    brez = int(sum(K[KODIRANE[c]].sum(axis=1).eq(0).sum() for c in KAT))
    kontrola.append({"zbirka": ime, "besedilnih stolpcev": len(KAT), "novih stolpcev 0/1": K.shape[1],
                     "vrednosti brez svojega stolpca": brez})
assert kodiranje == kodiranje_pred, "Uporaba pravil ne sme spreminjati naučenih kategorij."
ZNACILKE_ZA_MODEL = NUM + KODIRANE_VSE
display(pd.DataFrame(kontrola))
print("Značilk, pripravljenih za model:", len(ZNACILKE_ZA_MODEL), "=", len(NUM), "številskih +", len(KODIRANE_VSE), "stolpcev 0/1")

## Shranimo delitve, pravila in shemo
Izvozimo dopolnjene `train.csv`, `val.csv` in `test.csv`. Evidenca `delitev.csv`
ohrani ID-je in pripadnost zbirkam; `dopolnjevanje.json` vsebuje učne mediane;
`shema.json` ohrani dovoljene značilke, njihove skupine in razlago delitve.
Izvorni tabeli iz prvega poglavja ostaneta nespremenjeni.

Dodamo še stolpce 0/1 in `kodiranje.json` s seznamom kategorij. Shema dobi ključe
`znacilke_za_model`, `skupine_za_model` in `nabori_za_model`: v njih so besedilni stolpci
že zamenjani s stolpci 0/1, zato jih lahko podamo kateremukoli modelu. Dosedanji ključi ostanejo.

Pri poznejšem **prečnem preverjanju** mediane znova določimo znotraj vsakega
učnega dela. Zato bomo takrat iz prvotne tabele in evidence delitve obnovili
nedopolnjene učne vrstice ter uporabili Pipeline. Tukajšnji izvoz velja za
pripravljeno fiksno delitev train/val/test.

In [ ]:
# Vse tri datoteke imajo iste stolpce in tipe. Cilj in metapodatki niso del X.
stolpci_izhoda = META + [CILJ] + ZNACILKE + KODIRANE_VSE
koncne = {ime: pd.concat([df, kodirane[ime]], axis=1) for ime, df in zbirke.items()}
def za_model(stolpci):
    # Besedilni stolpec zamenjamo z njegovimi stolpci 0/1, vrstni red ostane.
    return [s for c in stolpci for s in KODIRANE.get(c, [c])]
izhodna_shema = copy.deepcopy(shema)
izhodna_shema.pop("datoteka", None)
izhodna_shema["razlicica"] = 2
izhodna_shema["vrstic"] = sum(len(df) for df in zbirke.values())
izhodna_shema["datoteke"] = {ime: f"{ime}.csv" for ime in zbirke}
izhodna_shema["vrstice_po_zbirkah"] = {ime: len(df) for ime, df in zbirke.items()}
izhodna_shema["izlocenih_vrstic"] = int(vloge.eq("izloceno").sum())
izhodna_shema["delitev"] = PARAMETRI_DELITVE
izhodna_shema["vhodna_tabela"] = str(pot_csv.relative_to(KOREN))
izhodna_shema["vhodna_shema"] = str(pot_sheme.relative_to(KOREN))
izhodna_shema["sha256_vhoda"] = hashlib.sha256(pot_csv.read_bytes()).hexdigest()
izhodna_shema["sha256_vhodne_sheme"] = hashlib.sha256(pot_sheme.read_bytes()).hexdigest()
izhodna_shema["dopolnjevanje"] = "dopolnjevanje.json"
izhodna_shema["kodiranje"] = "kodiranje.json"
izhodna_shema["kodirane_znacilke"] = KODIRANE
izhodna_shema["znacilke_za_model"] = ZNACILKE_ZA_MODEL
izhodna_shema["skupine_za_model"] = {k: za_model(v) for k, v in shema["skupine_znacilk"].items()}
izhodna_shema["nabori_za_model"] = {k: za_model(v) for k, v in shema["nabori_znacilk"].items()}
izhodna_shema["stevilo_znacilk_za_model"] = len(ZNACILKE_ZA_MODEL)
izhodna_shema["evidenca_delitve"] = "delitev.csv"
izhodna_shema["opombe_priprave_pred_delitvijo"] = izhodna_shema.pop("opombe")
izhodna_shema["opombe"] = [
    "Delitev je fiksna in določena pred dopolnjevanjem. Pravila so izključno iz train.",
    "Dopolnjene so samo značilke, cilj in metapodatki ostanejo nespremenjeni.",
    "Besedilne značilke so ohranjene in dodatno kodirane v stolpce 0/1; kategorije so izključno iz train.",
    "Značilke niso skalirane; napovedni model ni naučen.",
    "Test je tehnično pripravljen, vendar ga ne uporabljamo za izbor modelov ali nastavitev.",
    "Za CV obnovimo nedopolnjene učne vrstice iz vhoda; pripravo ponovno učimo znotraj vsakega kroga.",
]
opis_stolpca = {r["stolpec"]: r for r in izhodna_shema["stolpci"]}
for c, stolpci in KODIRANE.items():
    for s, k in zip(stolpci, kodiranje["stolpci"][c]["kategorije"] + [kodiranje["skupna_kategorija"]]):
        izhodna_shema["stolpci"].append({"stolpec": s, "vloga": "znacilka", "dtype": "int8", "manjkajocih": 0,
            "opis": f"1 = {c} je {k}; 0 = ni.", "vir": c, "skupina": opis_stolpca[c]["skupina"]})
for r in izhodna_shema["stolpci"]:
    r["manjkajocih_v_vhodu"] = r.pop("manjkajocih")
    r["dtype"] = str(koncne["train"][r["stolpec"]].dtype)
    r["manjkajocih_po_zbirkah"] = {ime: int(df[r["stolpec"]].isna().sum()) for ime, df in koncne.items()}

tipi_izhoda = {r["stolpec"]: r["dtype"] for r in izhodna_shema["stolpci"] if r["stolpec"] not in DATUMI}
for ime, df in koncne.items():
    ciljna_pot = IZHOD / f"{ime}.csv"
    df[stolpci_izhoda].to_csv(ciljna_pot, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
    ponovno = pd.read_csv(ciljna_pot, dtype=tipi_izhoda, parse_dates=DATUMI,
        keep_default_na=False, na_values=[""])
    pd.testing.assert_frame_equal(df[stolpci_izhoda].reset_index(drop=True), ponovno,
                                  check_exact=False, rtol=1e-10, atol=1e-10)
evidenca.to_csv(IZHOD / "delitev.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
for ime, vsebina in [("dopolnjevanje.json", pravila), ("kodiranje.json", kodiranje), ("shema.json", izhodna_shema)]:
    (IZHOD / ime).write_text(json.dumps(vsebina, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print("Izhod:", IZHOD.relative_to(KOREN))
print("Vrstice:", izhodna_shema["vrstice_po_zbirkah"], "izločeno:", izhodna_shema["izlocenih_vrstic"])
print("V vseh treh zbirkah:", len(ZNACILKE), "značilk brez manjkajočih vrednosti;",
      len(ZNACILKE_ZA_MODEL), "značilk, pripravljenih za model.")

## Predaja modeliranju
Spodaj pripravimo ločene X, y in metapodatke. V X so samo številske značilke in stolpci 0/1,
zato jih lahko podamo kateremukoli modelu. Vseh stolpcev CSV-ja ne podamo modelu:
ključ, cilj, metapodatki in besedilni stolpci niso vhod.

In [ ]:
X_train, y_train = koncne["train"][ZNACILKE_ZA_MODEL].copy(), koncne["train"][CILJ].copy()
X_val, y_val = koncne["val"][ZNACILKE_ZA_MODEL].copy(), koncne["val"][CILJ].copy()
X_test, y_test = koncne["test"][ZNACILKE_ZA_MODEL].copy(), koncne["test"][CILJ].copy()
meta_train = zbirke["train"][META].copy()
meta_val = zbirke["val"][META].copy()
meta_test = zbirke["test"][META].copy()
print("X_train:", X_train.shape, "X_val:", X_val.shape, "X_test:", X_test.shape)
# Za prvi model lahko izberemo isti osnovni nabor na vseh treh zbirkah.
osnovni = izhodna_shema["nabori_za_model"]["osnovni"]
print("Osnovni nabor:", osnovni)

### Za pozneje: obnovimo učne vrstice pred imputacijo
To ni nova delitev in ne izvaja prečnega preverjanja. Pokaže, kako iz shranjene
evidence vrnemo originalne učne vrstice za poznejši Pipeline. Kontrolni vsoti
preprečita, da bi tiho združili različici vhodnih podatkov in stare delitve.

In [ ]:
shranjena_shema = json.loads((IZHOD / "shema.json").read_text(encoding="utf-8"))
assert hashlib.sha256(pot_csv.read_bytes()).hexdigest() == shranjena_shema["sha256_vhoda"]
assert hashlib.sha256(pot_sheme.read_bytes()).hexdigest() == shranjena_shema["sha256_vhodne_sheme"]
shranjena_delitev = pd.read_csv(IZHOD / "delitev.csv", dtype={KLJUC: "string"})
ucni_id = shranjena_delitev.loc[shranjena_delitev["zbirka"].eq("train"), KLJUC]
X_train_pred_imputacijo = podatki.set_index(KLJUC).loc[ucni_id, ZNACILKE].copy()
y_train_pred_imputacijo = podatki.set_index(KLJUC).loc[ucni_id, CILJ].copy()
pd.testing.assert_frame_equal(X_train_pred_imputacijo,
    zbirke_pred["train"].set_index(KLJUC).loc[ucni_id, ZNACILKE])
print("Obnovljene učne vrstice:", len(X_train_pred_imputacijo))

Vira: [stratificirana delitev](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
in [preprečevanje uhajanja podatkov](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).